# PaySim Feature Engineering

Dataset: PaySim Fraud Detection

| Step | Note |
|------|------|
| Memory Optimization | Downcasts numeric data types to reduce memory usage |
| Data Loading | Loads raw PaySim transaction data and sorts by time |
| Baseline Cleaning | Encodes categorical transaction type and keeps raw baseline features |
| Domain-Specific Features | Adds transaction consistency, balance anomaly, and temporal behavior features |
| Output | Saves baseline and engineered datasets to Parquet files |

In [1]:
import pandas as pd
import numpy as np
import gc
import warnings
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

DATA_PATH = 'data_raw/PaySim/PS_20174392719_1491204439457_log.csv'

# 1. MEMORY OPTIMIZATION
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if str(col_type) in ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']:
            c_min, c_max = df[col].min(), df[col].max()
            
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Mem decreased to {end_mem:.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

# 2. LOAD DATA
print("Loading data...")
df = pd.read_csv(DATA_PATH)

# Sort by Time (Chronological Splitting Requirement)
df = df.sort_values('step').reset_index(drop=True)

n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)
df_train = df.iloc[:train_end].copy()
df_val   = df.iloc[train_end:val_end].copy()
df_test  = df.iloc[val_end:].copy()
print(f"Split: {len(df_train):,} train / {len(df_val):,} val / {len(df_test):,} test rows")
del df; gc.collect()

# 3. BASELINE CLEANING
print("Applying baseline cleaning...")

# Encode transaction type
le = LabelEncoder()
df_train['type'] = le.fit_transform(df_train['type'].astype(str))
df_val['type']   = le.transform(df_val['type'].astype(str))
df_test['type']  = le.transform(df_test['type'].astype(str))

# Drop high-cardinality identifier columns for ML baseline
for df_ in [df_train, df_val, df_test]:
    df_.drop(columns=['nameOrig', 'nameDest'], inplace=True, errors='ignore')

# Optimize and save baseline parquet
df_train_base = reduce_mem_usage(df_train.copy())
df_val_base   = reduce_mem_usage(df_val.copy())
df_test_base  = reduce_mem_usage(df_test.copy())
df_train_base.to_parquet('data/paysim_baseline_train.parquet', index=False)
df_val_base.to_parquet('data/paysim_baseline_val.parquet', index=False)
df_test_base.to_parquet('data/paysim_baseline_test.parquet', index=False)

print("Baseline dataset saved: data/paysim_baseline_{split}.parquet")

# 4. DOMAIN-SPECIFIC FEATURE ENGINEERING (PaySim-specific)
print("Adding Domain-Specific Features...")

df_train_eng = df_train_base.copy()
df_val_eng   = df_val_base.copy()
df_test_eng  = df_test_base.copy()

def add_engineered_features(df, amt_95):
    # A. TRANSACTION CONSISTENCY
    df['orig_error'] = df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
    df['dest_error'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
    # B. BALANCE CHANGE
    df['orig_balance_diff'] = df['oldbalanceOrg'] - df['newbalanceOrig']
    df['dest_balance_diff'] = df['newbalanceDest'] - df['oldbalanceDest']
    # C. RATIO FEATURES
    df['amt_to_oldbalance_org']  = df['amount'] / (df['oldbalanceOrg']  + 1)
    df['amt_to_oldbalance_dest'] = df['amount'] / (df['oldbalanceDest'] + 1)
    # D. ZERO BALANCE FLAGS
    df['orig_zero_before'] = (df['oldbalanceOrg']  == 0).astype(np.int8)
    df['orig_zero_after']  = (df['newbalanceOrig'] == 0).astype(np.int8)
    df['dest_zero_before'] = (df['oldbalanceDest'] == 0).astype(np.int8)
    df['dest_zero_after']  = (df['newbalanceDest'] == 0).astype(np.int8)
    # E. TEMPORAL
    df['hour'] = (df['step'] % 24).astype(np.int8)
    df['day']  = (df['step'] // 24).astype(np.int16)
    # F. LARGE TRANSACTION (threshold passed in from train)
    df['is_large_transaction'] = (df['amount'] > amt_95).astype(np.int8)
    # G. TYPE-AMOUNT INTERACTION
    df['type_amount_interaction'] = df['type'] * df['amount']
    df = df.replace([np.inf, -np.inf], np.nan).fillna(-999)
    return df

amt_95 = df_train_eng['amount'].quantile(0.95)  # computed on train only

df_train_eng = add_engineered_features(df_train_eng, amt_95)
df_val_eng   = add_engineered_features(df_val_eng,   amt_95)
df_test_eng  = add_engineered_features(df_test_eng,  amt_95)

# Fill any numerical anomalies
df_train_eng = df_train_eng.replace([np.inf, -np.inf], np.nan).fillna(-999)
df_val_eng = df_val_eng.replace([np.inf, -np.inf], np.nan).fillna(-999)
df_test_eng = df_test_eng.replace([np.inf, -np.inf], np.nan).fillna(-999)

# Optimize and save engineered parquet
reduce_mem_usage(df_train_eng).to_parquet('data/paysim_feature_train.parquet', index=False)
reduce_mem_usage(df_val_eng).to_parquet('data/paysim_feature_val.parquet', index=False)
reduce_mem_usage(df_test_eng).to_parquet('data/paysim_feature_test.parquet', index=False)

print("Process Complete. Produced: paysim_baseline and paysim_feature")

Loading data...
Split: 4,453,834 train / 954,393 val / 954,393 test rows
Applying baseline cleaning...
Mem decreased to 106.19 Mb (65.3% reduction)
Mem decreased to 22.75 Mb (65.3% reduction)
Mem decreased to 22.75 Mb (65.3% reduction)
Baseline dataset saved: data/paysim_baseline_{split}.parquet
Adding Domain-Specific Features...
Mem decreased to 254.85 Mb (1.6% reduction)
Mem decreased to 54.61 Mb (1.6% reduction)
Mem decreased to 54.61 Mb (1.6% reduction)
Process Complete. Produced: paysim_baseline and paysim_feature
